# Surveillance Video Analysis — ResNet18 + LSTM
**Module:** ST7088CEM – Artificial Neural Networks | **Student:** Manjil Karki

**Architecture:** Pretrained ResNet18 backbone (ImageNet) + LSTM temporal head  
**Anomaly score:** `1 − P(Normal)` — no reconstruction loss  
**Two-phase training:** Phase 1 freezes backbone (head only), Phase 2 unfreezes layer4  
**Dataset:** UCF-Crime pre-extracted 64×64 PNGs — `data/dataset/Train/<class>/` and `data/dataset/Test/<class>/`

In [ ]:
# ── Path detection: works whether you run from project root or notebooks/ ──
import os
from pathlib import Path

def _find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for c in candidates:
        if (c / 'data' / 'dataset').exists():
            return c
    raise FileNotFoundError(
        f'Cannot find data/dataset from {Path.cwd()}. '
        f'Place this notebook inside the project that has data/dataset/ at its root.')

_ROOT = _find_project_root()
os.chdir(_ROOT)
print(f'Project root: {_ROOT}')

# ── Config ──────────────────────────────────────────────────────────────────
import torch, random
import numpy as np

DATASET_ROOT  = Path('data/dataset')
OUTPUTS_DIR   = Path('outputs_resnet')
MODELS_DIR    = OUTPUTS_DIR / 'models'
PLOTS_DIR     = OUTPUTS_DIR / 'plots'
RESULTS_DIR   = OUTPUTS_DIR / 'results'
for d in (MODELS_DIR, PLOTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    'NormalVideos': 'Normal', 'RoadAccidents': 'RoadAccidents',
    'Shoplifting': 'Shoplifting', 'Arson': 'Arson', 'Burglary': 'Burglary',
}
CLASSES         = list(CLASS_MAP.values())   # ['Normal','RoadAccidents','Shoplifting','Arson','Burglary']
NUM_CLASSES     = len(CLASSES)
CLASS_TO_IDX    = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS    = {i: c for c, i in CLASS_TO_IDX.items()}
ANOMALY_CLASSES = [c for c in CLASSES if c != 'Normal']
_CLASS_COLORS   = {
    'Normal': '#4FC3F7', 'RoadAccidents': '#29B6F6',  # sky blue — road
    'Shoplifting': '#66BB6A', 'Arson': '#FF7043', 'Burglary': '#AB47BC',
}

FRAME_H = FRAME_W = 112
CLIP_LEN      = 16          # 16 frames → 1.6 sec at 10fps
TRAIN_STRIDE  = 8           # 50% overlap for train clips
TEST_STRIDE   = 4           # 75% overlap for val/test clips
VAL_SPLIT     = 0.25        # 25% of source videos → validation

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Model ───────────────────────────────────────────────────────────────────
LATENT_DIM    = 512         # ResNet18 avgpool output dim
LSTM_UNITS    = 256
FC_UNITS      = 128
DROPOUT       = 0.5

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE    = 32          # reduced for 112×112 VRAM budget
EVAL_BS       = 128
NUM_WORKERS   = 12          # more workers for 50 GB RAM
PHASE1_EPOCHS = 30          # head only (backbone frozen)
PHASE2_EPOCHS = 20          # layer4 + head (differential LR)
LR_HEAD       = 1e-3
LR_BACKBONE   = 1e-5        # 100× lower for layer4 fine-tune
PHASE2_LR_HEAD = 3e-4       # lower head LR in phase2 to stop oscillation
WEIGHT_DECAY  = 1e-4
N_PER_CLASS   = 5000        # WeightedRandomSampler draws N_PER_CLASS × NUM_CLASSES clips/epoch
SEED          = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU    : {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')
print(f'Outputs: {OUTPUTS_DIR.resolve()}')

In [ ]:
import json, random
from collections import defaultdict

import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_fscore_support, f1_score,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print('Imports OK')

In [ ]:
# ── Data discovery helpers ────────────────────────────────────────────────────

def _parse_video_id(stem):
    """'RoadAccidents002_x264_1020' → ('RoadAccidents002_x264', 1020)"""
    parts = stem.rsplit('_', 1)
    return (parts[0], int(parts[1])) if len(parts) == 2 else (stem, 0)

def discover_clips(root, stride=TEST_STRIDE, clip_len=CLIP_LEN):
    """Enumerate all clips in a dataset root (no train/val split)."""
    clips, labels = [], []
    for folder in sorted(Path(root).iterdir()):
        if not folder.is_dir(): continue
        cls_name = CLASS_MAP.get(folder.name)
        if cls_name is None: continue
        videos = defaultdict(list)
        for img in sorted(folder.glob('*.png')):
            vid_id, frame_num = _parse_video_id(img.stem)
            videos[vid_id].append((frame_num, img))
        for vid_id, frames in sorted(videos.items()):
            frames = [f[1] for f in sorted(frames)]
            if len(frames) < clip_len: continue
            for s in range(0, len(frames) - clip_len + 1, stride):
                clips.append(frames[s:s + clip_len])
                labels.append(CLASS_TO_IDX[cls_name])
    return clips, np.array(labels)

def make_video_splits(root, val_split=VAL_SPLIT,
                      train_stride=TRAIN_STRIDE, test_stride=TEST_STRIDE,
                      clip_len=CLIP_LEN):
    """Video-level train/val split to prevent temporal data leakage."""
    tr_clips, tr_lbl, va_clips, va_lbl = [], [], [], []
    for folder in sorted(Path(root).iterdir()):
        if not folder.is_dir(): continue
        cls_name = CLASS_MAP.get(folder.name)
        if cls_name is None: continue
        cls_idx = CLASS_TO_IDX[cls_name]
        videos = defaultdict(list)
        for img in sorted(folder.glob('*.png')):
            vid_id, frame_num = _parse_video_id(img.stem)
            videos[vid_id].append((int(frame_num), img))
        vid_ids = sorted(videos.keys()); random.shuffle(vid_ids)
        n_val   = max(1, int(len(vid_ids) * val_split))
        val_set = set(vid_ids[:n_val])
        for vid_id, frames in videos.items():
            frames = [f[1] for f in sorted(frames)]
            if len(frames) < clip_len: continue
            stride = test_stride if vid_id in val_set else train_stride
            for s in range(0, len(frames) - clip_len + 1, stride):
                clip = frames[s:s + clip_len]
                if vid_id in val_set:
                    va_clips.append(clip); va_lbl.append(cls_idx)
                else:
                    tr_clips.append(clip); tr_lbl.append(cls_idx)
    return tr_clips, np.array(tr_lbl), va_clips, np.array(va_lbl)

print('Data helpers defined.')

In [ ]:
# ── Dataset & DataLoader ──────────────────────────────────────────────────────

_train_tf = T.Compose([
    T.RandomResizedCrop(FRAME_H, scale=(0.7, 1.0), ratio=(1.0, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
_val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def _motion_augment(raw):
    """In-place temporal augmentations on a list of PIL frames."""
    # Temporal reverse
    if random.random() < 0.5:
        raw = raw[::-1]
    # Frame dropout: replace one inner frame with its left neighbour
    if random.random() < 0.3 and len(raw) > 2:
        i = random.randint(1, len(raw) - 2)
        raw = list(raw)
        raw[i] = raw[i - 1]
    # Speed perturbation: randomly repeat one frame (simulates slow-motion)
    if random.random() < 0.2 and len(raw) > 2:
        i = random.randint(0, len(raw) - 2)
        raw = list(raw)
        raw[i + 1] = raw[i]
    return raw

class ClipDataset(Dataset):
    def __init__(self, clips, labels=None, training=False):
        self.clips    = clips
        self.labels   = labels
        self.training = training

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        frame_paths = self.clips[idx]
        raw = []
        for p in frame_paths:
            img = Image.open(p).convert('RGB')
            if img.size != (FRAME_W, FRAME_H):
                img = img.resize((FRAME_W, FRAME_H), Image.BILINEAR)
            raw.append(img)
        if self.training:
            raw = _motion_augment(raw)
        # Shared spatial seed → same crop/flip applied to all frames in clip
        seed = random.randint(0, 2**31)
        tensors = []
        tf = _train_tf if self.training else _val_tf
        for img in raw:
            if self.training:
                random.seed(seed); torch.manual_seed(seed)
            tensors.append(tf(img))
        # Gaussian noise on tensor (motion textures, not colour)
        x = torch.stack(tensors)  # (T, 3, H, W)
        if self.training and random.random() < 0.2:
            x = x + torch.randn_like(x) * 0.05
        if self.labels is not None:
            return x, int(self.labels[idx])
        return x

def make_loader(clips, labels=None, training=False, bs=BATCH_SIZE):
    ds = ClipDataset(clips, labels, training=training)
    sampler = None
    if training and labels is not None:
        counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
        counts  = np.maximum(counts, 1)
        weights = torch.from_numpy(1.0 / counts[labels]).float()
        sampler = WeightedRandomSampler(
            weights, num_samples=N_PER_CLASS * NUM_CLASSES, replacement=True)
    return DataLoader(
        ds, batch_size=bs,
        shuffle=(training and sampler is None), sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

print('Dataset helpers defined  (motion augmentations: reverse 50%, dropout 30%, speed 20%, noise 20%).')

In [ ]:
# ── Model Architecture ────────────────────────────────────────────────────────

class ResNetEncoder(nn.Module):
    """TimeDistributed ResNet18: (B, T, 3, H, W) → (B, T, 512)."""
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = resnet18(weights=weights)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

    def forward(self, x):
        B, T, C, H, W = x.shape
        f = self.features(x.view(B * T, C, H, W))   # (B*T, 512, 1, 1)
        return f.view(B, T, -1)                       # (B, T, 512)

    def freeze_all(self):
        for p in self.parameters(): p.requires_grad = False

    def unfreeze_layer4(self):
        for name, p in self.named_parameters():
            if 'features.7' in name:
                p.requires_grad = True

    def unfreeze_last_block(self):
        for name, p in self.named_parameters():
            if 'features.7.1' in name:
                p.requires_grad = True

    def unfreeze_layer3_4(self):
        for name, p in self.named_parameters():
            if 'features.6' in name or 'features.7' in name:
                p.requires_grad = True


class TemporalAttentionHead(nn.Module):
    """Single-layer transformer encoder with CLS token: (B,T,512) → (B,NUM_CLASSES).

    CLS token attends over all T frame embeddings via multi-head self-attention,
    then a FFN projects it to class logits.  More parallelisable than LSTM and
    can attend directly to the most anomalous frames.
    """
    def __init__(self):
        super().__init__()
        self.cls  = nn.Parameter(torch.zeros(1, 1, LATENT_DIM))
        self.pos  = nn.Parameter(torch.zeros(1, CLIP_LEN + 1, LATENT_DIM))
        self.attn = nn.MultiheadAttention(LATENT_DIM, num_heads=8,
                                           dropout=0.1, batch_first=True)
        self.norm1 = nn.LayerNorm(LATENT_DIM)
        self.norm2 = nn.LayerNorm(LATENT_DIM)
        self.ffn  = nn.Sequential(
            nn.Linear(LATENT_DIM, LATENT_DIM * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(LATENT_DIM * 2, LATENT_DIM),
        )
        self.fc   = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(LATENT_DIM, FC_UNITS),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FC_UNITS, NUM_CLASSES),
        )
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):                              # x: (B, T, 512)
        B = x.size(0)
        cls = self.cls.expand(B, -1, -1)               # (B, 1, 512)
        x   = torch.cat([cls, x], dim=1)               # (B, T+1, 512)
        x   = x + self.pos[:, :x.size(1)]              # learned positional encoding
        a, _ = self.attn(x, x, x)
        x   = self.norm1(x + a)                        # residual + LN
        x   = self.norm2(x + self.ffn(x))              # FFN + LN
        return self.fc(x[:, 0])                        # CLS token → logits


class LSTMHead(nn.Module):
    """(B, T, 512) → (B, NUM_CLASSES). Uses last hidden state."""
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(LATENT_DIM, LSTM_UNITS, batch_first=True)
        self.fc   = nn.Sequential(
            nn.Linear(LSTM_UNITS, FC_UNITS), nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FC_UNITS, NUM_CLASSES),
        )
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])


class TemporalPoolHead(nn.Module):
    """Ablation baseline: mean-pool over T then FC."""
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(LATENT_DIM, FC_UNITS), nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FC_UNITS, NUM_CLASSES),
        )
    def forward(self, x):
        return self.fc(x.mean(dim=1))


class VideoClassifier(nn.Module):
    def __init__(self, head):
        super().__init__()
        self.encoder = ResNetEncoder(pretrained=True)
        self.head    = head

    def forward(self, x):
        return self.head(self.encoder(x))


_enc = ResNetEncoder(); _h = TemporalAttentionHead()
print(f'ResNetEncoder params          : {sum(p.numel() for p in _enc.parameters()):,}')
print(f'TemporalAttentionHead params  : {sum(p.numel() for p in _h.parameters()):,}')
del _enc, _h
print('Models defined.')

In [ ]:
# ── FocalLoss ─────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """Down-weights easy examples so hard-to-separate classes get more gradient."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt   = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def _make_criterion(device):
    # Uniform class weights — these classes are visually distinct enough not to need per-class boosting.
    # gamma=2 still focuses gradient on hard examples within each class.
    return FocalLoss(alpha=None, gamma=2.0)


# ── Training loop ─────────────────────────────────────────────────────────────

_SCALER = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

def _epoch_loop(model, loader, opt, ce, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0; correct = 0; total = 0
    all_preds = []; all_labels = []
    amp_enabled = torch.cuda.is_available()
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, labels in loader:
            x      = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if train: opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                logits = model(x)
                loss   = ce(logits, labels)
            if train:
                _SCALER.scale(loss).backward()
                _SCALER.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                _SCALER.step(opt)
                _SCALER.update()
            total_loss += loss.item()
            preds = logits.argmax(1)
            correct += (preds == labels).sum().item(); total += len(labels)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    acc      = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), acc, macro_f1, all_labels, all_preds

def _per_cls_str(labels, preds):
    correct = [0] * NUM_CLASSES; total = [0] * NUM_CLASSES
    for l, p in zip(labels, preds):
        correct[l] += int(l == p); total[l] += 1
    return '  '.join(
        f'{IDX_TO_CLASS[i][0]}:{correct[i]/max(total[i],1):.2f}'
        for i in range(NUM_CLASSES))

def train_phase(model, tr_loader, va_loader, opt, sched,
                epochs, phase_name, save_path, device=DEVICE,
                patience=7):
    ce      = _make_criterion(device)
    best_f1 = 0.0; no_imp = 0
    hist    = {k: [] for k in ['train_loss', 'train_acc', 'train_f1',
                                'val_loss',   'val_acc',   'val_f1']}
    for epoch in range(epochs):
        tr_l, tr_a, tr_f, _, _          = _epoch_loop(model, tr_loader, opt, ce, device, True)
        sched.step()
        va_l, va_a, va_f, va_lbl, va_pd = _epoch_loop(model, va_loader, opt, ce, device, False)
        for k, v in zip(hist.keys(), [tr_l, tr_a, tr_f, va_l, va_a, va_f]):
            hist[k].append(float(v))
        if va_f > best_f1:
            best_f1 = va_f; no_imp = 0
            torch.save(model.state_dict(), save_path)
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'Early stop @ epoch {epoch+1}'); break
        print(f'{phase_name} {epoch+1:02d}/{epochs}  '
              f'loss={tr_l:.4f} acc={tr_a:.3f} f1={tr_f:.3f} | '
              f'val_loss={va_l:.4f} val_acc={va_a:.3f} val_f1={va_f:.3f}  '
              f'[{_per_cls_str(va_lbl, va_pd)}]')
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    print(f'{phase_name} done. Best val macro F1: {best_f1:.4f}')
    return hist


def plot_training(hist, title_prefix, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    pairs = [('train_loss', 'val_loss',  'Loss',     'Loss'),
             ('train_acc',  'val_acc',   'Accuracy', 'Accuracy'),
             ('train_f1',   'val_f1',    'Macro F1', 'Macro F1 (checkpoint metric)')]
    for ax, (tk, vk, yl, title) in zip(axes, pairs):
        ax.plot(hist[tk], label='Train', lw=1.5)
        ax.plot(hist[vk], label='Val',   lw=1.5, ls='--')
        if 'f1' in vk and hist[vk]:
            best = int(np.argmax(hist[vk]))
            ax.axvline(best, color='gray', ls=':', lw=1, label=f'Best ep={best+1}')
        ax.set_xlabel('Epoch'); ax.set_ylabel(yl)
        ax.set_title(f'{title_prefix} — {title}', fontweight='bold')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

print('Training functions defined  (AMP enabled when CUDA available).')

In [ ]:
# ── Evaluation helpers ────────────────────────────────────────────────────────

def _make_eval_loader(clips, labels=None, bs=EVAL_BS):
    ds = ClipDataset(clips, labels, training=False)
    return DataLoader(ds, batch_size=bs, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

def get_anomaly_scores(model, clips, device=DEVICE):
    model.eval(); scores = []
    with torch.no_grad():
        for x in _make_eval_loader(clips):
            probs = torch.softmax(model(x.to(device)), dim=1)
            scores.extend((1.0 - probs[:, CLASS_TO_IDX['Normal']]).cpu().tolist())
    return np.array(scores)

def get_predictions(model, clips, labels=None, device=DEVICE):
    model.eval(); all_preds = []; all_probs = []
    ldr = _make_eval_loader(clips, labels)
    with torch.no_grad():
        for batch in ldr:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            logits = model(x.to(device))
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.array(all_preds), np.concatenate(all_probs, 0)

def get_embeddings(model, clips, device=DEVICE):
    model.eval(); embs = []
    with torch.no_grad():
        for x in _make_eval_loader(clips):
            feat = model.encoder(x.to(device))   # (B, T, 512)
            embs.append(feat.mean(dim=1).cpu().numpy())
    return np.concatenate(embs, 0)

def _youden_threshold(scores, binary_gt):
    fpr, tpr, thresholds = roc_curve(binary_gt, scores)
    return float(thresholds[np.argmax(tpr - fpr)])

def threshold_comparison(model, val_normal_clips, test_clips, y_test, device=DEVICE):
    n_sc = get_anomaly_scores(model, val_normal_clips, device)
    t_sc = get_anomaly_scores(model, test_clips,       device)
    bgt  = (y_test != CLASS_TO_IDX['Normal']).astype(int)
    auc  = roc_auc_score(bgt, t_sc)
    results = {}
    for name, thr in [
        ('Mean+2std', float(n_sc.mean() + 2 * n_sc.std())),
        ('Mean+3std', float(n_sc.mean() + 3 * n_sc.std())),
        ('ROC-Opt',   _youden_threshold(t_sc, bgt)),
    ]:
        preds = (t_sc > thr).astype(int)
        p, r, f, _ = precision_recall_fscore_support(bgt, preds, average='binary', zero_division=0)
        results[name] = {'threshold': thr, 'precision': p, 'recall': r, 'f1': f, 'auc': auc}
    print(f"\n{'Method':<12} {'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AUC':>8}")
    print('-' * 58)
    for n, r in results.items():
        print(f"{n:<12} {r['threshold']:>10.5f} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1']:>8.3f} {r['auc']:>8.3f}")
    return results, n_sc, t_sc, bgt

def _save_json(data, path):
    with open(path, 'w') as f: json.dump(data, f, indent=2, default=float)
    print(f'[OK] {path}')

def plot_roc(scores, binary_gt, title, save_path):
    fpr, tpr, _ = roc_curve(binary_gt, scores); auc = roc_auc_score(binary_gt, scores)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f'AUC={auc:.3f}'); ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title(title, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

def plot_anomaly_scores(scores, labels, threshold, save_path):
    fig, ax = plt.subplots(figsize=(12, 3.5)); x = np.arange(len(scores))
    nm = labels == CLASS_TO_IDX['Normal']; an = ~nm
    ax.scatter(x[nm], scores[nm], s=6, alpha=0.5, color='steelblue', label='Normal')
    ax.scatter(x[an], scores[an], s=6, alpha=0.7, color='crimson',   label='Anomaly')
    ax.axhline(threshold, color='darkorange', ls='--', lw=1.5, label=f'Thr={threshold:.4f}')
    ax.set_xlabel('Clip'); ax.set_ylabel('1 − P(Normal)')
    ax.set_title('Per-Clip Anomaly Scores', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

def plot_confusion_matrix(y_true, y_pred, save_path):
    cm_  = confusion_matrix(y_true, y_pred)
    cm_n = cm_.astype(float) / cm_.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, data, fmt, title in [(axes[0], cm_, 'd', 'Counts'), (axes[1], cm_n, '.2f', 'Normalised')]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(title, fontweight='bold')
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        ax.tick_params(axis='y', rotation=0,  labelsize=8)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

def plot_per_class_metrics(report, save_path):
    metrics = {cls: report[cls] for cls in CLASSES if cls in report}
    x = np.arange(len(metrics)); w = 0.25
    prec = [metrics[c]['precision'] for c in metrics]
    rec  = [metrics[c]['recall']    for c in metrics]
    f1   = [metrics[c]['f1-score']  for c in metrics]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x-w, prec, w, label='Precision', color='#1E88E5')
    ax.bar(x,   rec,  w, label='Recall',    color='#43A047')
    ax.bar(x+w, f1,   w, label='F1',        color='#FB8C00')
    ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()), rotation=20, ha='right')
    ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
    ax.set_title('Per-Class Metrics', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

def plot_latent_space(embeddings, labels, method='pca', save_path=None):
    reducer = (TSNE(n_components=2, random_state=42, perplexity=30)
               if method == 'tsne' else PCA(n_components=2, random_state=42))
    coords = reducer.fit_transform(embeddings)
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, cls in enumerate(CLASSES):
        mask = labels == i
        ax.scatter(coords[mask, 0], coords[mask, 1], s=12, alpha=0.6,
                   color=_CLASS_COLORS.get(cls, '#888'), label=cls)
    ax.set_title(f"{'t-SNE' if method=='tsne' else 'PCA'} of Clip Embeddings", fontweight='bold')
    ax.legend(fontsize=9, markerscale=2); ax.grid(alpha=0.3); plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight'); print(f'[OK] {save_path}')
    plt.close('all')

print('Evaluation helpers defined.')

In [ ]:
# ── Timeline helpers ──────────────────────────────────────────────────────────

def build_test_sequence(test_clips, y_test, clips_per_segment=20):
    seq_clips, seq_labels, segments = [], [], []
    norm_idx = np.where(y_test == CLASS_TO_IDX['Normal'])[0].copy()
    np.random.shuffle(norm_idx)
    offset = 0
    norm_cursor = [0]  # avoids reusing the same Normal clips

    def _add_normal(n):
        nonlocal offset
        start = norm_cursor[0]
        end   = min(start + n, len(norm_idx))
        chosen = norm_idx[start:end]
        norm_cursor[0] = end
        for i in chosen:
            seq_clips.append(test_clips[i]); seq_labels.append(CLASS_TO_IDX['Normal'])
        segments.append(('Normal', offset, offset + len(chosen) - 1))
        offset += len(chosen)

    _add_normal(clips_per_segment)
    for cls in ANOMALY_CLASSES:
        idx = np.where(y_test == CLASS_TO_IDX[cls])[0]
        if len(idx) == 0: continue
        chosen = idx[:clips_per_segment]
        for i in chosen:
            seq_clips.append(test_clips[i]); seq_labels.append(CLASS_TO_IDX[cls])
        segments.append((cls, offset, offset + len(chosen) - 1))
        offset += len(chosen)
        _add_normal(clips_per_segment // 2)

    return seq_clips, np.array(seq_labels), segments

def sliding_window_predict(model, seq_clips, device=DEVICE):
    model.eval(); all_preds = []; all_scores = []
    ldr = DataLoader(ClipDataset(seq_clips, training=False),
                     batch_size=EVAL_BS, shuffle=False, num_workers=0)
    with torch.no_grad():
        for x in ldr:
            logits = model(x.to(device))
            probs  = torch.softmax(logits, dim=1)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_scores.extend((1.0 - probs[:, CLASS_TO_IDX['Normal']]).cpu().tolist())
    return all_preds, all_scores

def majority_vote_smooth(preds, window=3):
    out = list(preds)
    for i in range(len(preds)):
        s = max(0, i - window // 2); e = min(len(preds), i + window // 2 + 1)
        vals = preds[s:e]; out[i] = max(set(vals), key=vals.count)
    return out

def merge_segments(smoothed, scores, threshold):
    if not smoothed: return []
    segs = []; cur_cls, cur_s, cur_sc = smoothed[0], 0, [scores[0]]
    for i in range(1, len(smoothed)):
        if smoothed[i] == cur_cls:
            cur_sc.append(scores[i])
        else:
            ms = float(np.mean(cur_sc))
            segs.append({'label': IDX_TO_CLASS[cur_cls], 'start': cur_s, 'end': i-1,
                         'mean_score': ms,
                         'anomaly': (cur_cls != CLASS_TO_IDX['Normal'] and ms > threshold)})
            cur_cls, cur_s, cur_sc = smoothed[i], i, [scores[i]]
    ms = float(np.mean(cur_sc))
    segs.append({'label': IDX_TO_CLASS[cur_cls], 'start': cur_s, 'end': len(smoothed)-1,
                 'mean_score': ms,
                 'anomaly': (cur_cls != CLASS_TO_IDX['Normal'] and ms > threshold)})
    return segs

def print_timeline(segments):
    print(f"\n{'Clips':>9}  {'Class':<12}  {'Score':>6}  Flag")
    print('-' * 42)
    for s in segments:
        flag = '⚠ ANOMALY' if s['anomaly'] else ''
        print(f"  {s['start']:>3}–{s['end']:<3}  {s['label']:<12}  {s['mean_score']:.3f}  {flag}")

def plot_score_over_time(scores, preds, threshold, save_path):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    ax1.plot(scores, lw=1.2, color='steelblue', label='Anomaly score')
    ax1.axhline(threshold, color='darkorange', ls='--', lw=1.2, label=f'Thr={threshold:.3f}')
    ax1.set_ylabel('1−P(Normal)'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
    ax1.set_title('Anomaly Score Over Synthetic Sequence', fontweight='bold')
    colors = [_CLASS_COLORS.get(IDX_TO_CLASS[p], '#888') for p in preds]
    ax2.bar(range(len(preds)), [1]*len(preds), color=colors, width=1.0)
    patches = [mpatches.Patch(color=v, label=k) for k, v in _CLASS_COLORS.items()]
    ax2.legend(handles=patches, fontsize=8, loc='upper right', ncol=5)
    ax2.set_xlabel('Clip index'); ax2.set_yticks([]); ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

def plot_timeline(segments, save_path):
    fig, ax = plt.subplots(figsize=(14, 3))
    for s in segments:
        color = _CLASS_COLORS.get(s['label'], '#888'); width = s['end'] - s['start'] + 1
        ax.barh(0, width, left=s['start'], height=0.6, color=color, alpha=0.85)
        if width > 3:
            ax.text(s['start'] + width/2, 0,
                    f"{s['label']}\n{'⚠' if s['anomaly'] else ''}",
                    ha='center', va='center', fontsize=7, fontweight='bold')
    patches = [mpatches.Patch(color=v, label=k) for k, v in _CLASS_COLORS.items()]
    ax.legend(handles=patches, fontsize=9, loc='upper right', ncol=5)
    ax.set_xlabel('Clip index'); ax.set_yticks([])
    ax.set_title('Predicted Event Timeline', fontweight='bold')
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'[OK] {save_path}')

print('Timeline helpers defined.')

In [ ]:
# ── Grad-CAM helpers ──────────────────────────────────────────────────────────

def compute_gradcam(model, clip_tensor, class_idx, frame_idx=None, device=DEVICE):
    """Grad-CAM on ResNet18 layer3[-1].conv2 (4×4 maps at 64×64 input).

    Uses a tensor hook on the activation output rather than register_full_backward_hook
    — more reliable across PyTorch versions.
    """
    if frame_idx is None: frame_idx = CLIP_LEN // 2
    model = model.to(device); model.eval()

    target = model.encoder.features[6][-1].conv2   # layer3 last block conv2
    acts = {}; grads = {}

    def _fwd_hook(m, inp, out):
        acts['f'] = out                                   # keep in graph (not detached)
        out.register_hook(lambda g: grads.update({'g': g.detach()}))

    h1 = target.register_forward_hook(_fwd_hook)

    x = clip_tensor.unsqueeze(0).to(device)
    with torch.enable_grad():
        logits = model(x)
        model.zero_grad()
        logits[0, class_idx].backward()
    h1.remove()

    act  = acts['f'][frame_idx].detach()   # (256, 4, 4)
    grad = grads['g'][frame_idx]            # (256, 4, 4)
    w    = grad.mean(dim=(1, 2))
    cam  = torch.relu((w[:, None, None] * act).sum(0))   # (4, 4)
    cam  = cam - cam.min(); cam = cam / (cam.max() + 1e-8)
    cam_np = cam.cpu().numpy()
    cam_up = np.array(Image.fromarray((cam_np * 255).astype(np.uint8)).resize(
        (FRAME_W, FRAME_H), Image.BILINEAR)) / 255.0
    pred_cls = int(logits.argmax(1).item())
    conf     = float(torch.softmax(logits, dim=1)[0, class_idx].item())
    return cam_up, pred_cls, conf


def overlay_heatmap(frame_rgb, heatmap, alpha=0.5):
    r = heatmap; g = np.zeros_like(r); b = np.zeros_like(r)
    colored = np.stack([r, g, b], axis=-1)
    return np.clip(frame_rgb * (1 - alpha) + colored * alpha, 0, 1)

def _denorm_frame(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()

def visualize_gradcam_batch(model, test_clips, y_test, n_per_class=2,
                             save_path=None, device=DEVICE):
    rows = []
    for cls_name in ANOMALY_CLASSES:
        cls_idx = CLASS_TO_IDX[cls_name]
        idxs    = np.where(y_test == cls_idx)[0][:n_per_class]
        for i in idxs:
            clip_t   = ClipDataset([test_clips[i]], training=False)[0]
            frame_np = _denorm_frame(clip_t[CLIP_LEN // 2])
            cam, pred, conf = compute_gradcam(
                model, clip_t, cls_idx, frame_idx=CLIP_LEN // 2, device=device)
            rows.append((frame_np, overlay_heatmap(frame_np, cam), cls_name, pred, conf))

    n = len(rows)
    fig, axes = plt.subplots(n, 2, figsize=(7, n * 2.5))
    if n == 1: axes = [axes]
    for row_ax, (frame, overlay, cls_name, pred, conf) in zip(axes, rows):
        row_ax[0].imshow(frame);        row_ax[0].set_title(f'True: {cls_name}', fontsize=8)
        row_ax[1].imshow(overlay)
        row_ax[1].set_title(f'Grad-CAM | pred={IDX_TO_CLASS[pred]} conf={conf:.2f}', fontsize=8)
        for ax in row_ax: ax.axis('off')
    plt.suptitle('Grad-CAM on ResNet18 Layer3', fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight'); print(f'[OK] {save_path}')
    plt.close('all')

print('Grad-CAM helpers defined  (tensor-hook approach).')

## Data Discovery
Scans Train/ and Test/ folders, groups frames by video ID, builds clips.

In [ ]:
random.seed(SEED); np.random.seed(SEED)

print('Building video-level train/val splits (this takes a few minutes)...')
train_clips, y_train, val_clips, y_val = make_video_splits(
    DATASET_ROOT / 'Train',
    val_split=VAL_SPLIT,
    train_stride=TRAIN_STRIDE,
    test_stride=TEST_STRIDE,
)
print('Loading test set...')
test_clips, y_test = discover_clips(DATASET_ROOT / 'Test', stride=TEST_STRIDE)

print(f'\nTrain: {len(train_clips):,}  Val: {len(val_clips):,}  Test: {len(test_clips):,}')
for split_name, lbls in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'\n{split_name}:')
    for cls in CLASSES:
        n = (lbls == CLASS_TO_IDX[cls]).sum()
        print(f'  {cls:<12}: {n:,}')

val_normal_clips = [val_clips[i] for i in range(len(val_clips))
                    if y_val[i] == CLASS_TO_IDX['Normal']]

tr_loader = make_loader(train_clips, y_train, training=True)
va_loader = make_loader(val_clips,   y_val,   training=False, bs=EVAL_BS)
print(f'\nTrain batches/epoch (sampler {N_PER_CLASS}×{NUM_CLASSES}={N_PER_CLASS*NUM_CLASSES}): {len(tr_loader)}')
print(f'Val   batches : {len(va_loader)}')

## Phase 1 — Backbone Frozen, LSTM Head Only (30 epochs)

In [ ]:
print('=' * 60 + '\nPHASE 1 — Backbone frozen, LSTM head only\n' + '=' * 60)
model = VideoClassifier(head=TemporalAttentionHead()).to(DEVICE)
model.encoder.freeze_all()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params (attention head only): {n_trainable:,}')

opt1   = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)

hist1 = train_phase(
    model, tr_loader, va_loader, opt1, sched1,
    epochs=PHASE1_EPOCHS, phase_name='P1',
    save_path=MODELS_DIR / 'phase1_best.pth',
    patience=7,
)
print(f'\nPhase 1 best val macro F1 : {max(hist1["val_f1"]):.4f}')
print(f'Phase 1 best val accuracy  : {max(hist1["val_acc"]):.4f}')

In [ ]:
plot_training(hist1, 'Phase 1', PLOTS_DIR / 'phase1_training.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'phase1_training.png')); plt.axis('off'); plt.show()

## Phase 2 — Unfreeze Layer4, Differential LR (20 epochs)

In [ ]:
print('=' * 60 + '\nPHASE 2a — Gradual: last block of layer4 only (8 epochs)\n' + '=' * 60)
model.encoder.unfreeze_last_block()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params (last layer4 block + head): {n_trainable:,}')

last_block_params = [p for n, p in model.encoder.named_parameters()
                     if 'features.7.1' in n and p.requires_grad]
head_params       = list(model.head.parameters())
opt2a   = torch.optim.Adam(
    [{'params': last_block_params, 'lr': LR_BACKBONE},
     {'params': head_params,       'lr': PHASE2_LR_HEAD}],
    weight_decay=WEIGHT_DECAY)
sched2a = torch.optim.lr_scheduler.CosineAnnealingLR(opt2a, T_max=8)

hist2a = train_phase(
    model, tr_loader, va_loader, opt2a, sched2a,
    epochs=8, phase_name='P2a',
    save_path=MODELS_DIR / 'phase2a_best.pth',
    patience=5,
)

print()
print('=' * 60 + '\nPHASE 2b — Full: layer3 + layer4 + head (17 epochs)\n' + '=' * 60)
model.encoder.unfreeze_layer3_4()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params (layer3+layer4 + head): {n_trainable:,}')

layer4_params = [p for n, p in model.encoder.named_parameters()
                 if 'features.7' in n and p.requires_grad]
layer3_params = [p for n, p in model.encoder.named_parameters()
                 if 'features.6' in n and p.requires_grad]
head_params   = list(model.head.parameters())
opt2b   = torch.optim.Adam(
    [{'params': layer3_params, 'lr': LR_BACKBONE * 0.1},
     {'params': layer4_params, 'lr': LR_BACKBONE},
     {'params': head_params,   'lr': PHASE2_LR_HEAD}],
    weight_decay=WEIGHT_DECAY)
sched2b = torch.optim.lr_scheduler.CosineAnnealingLR(opt2b, T_max=17)

hist2 = train_phase(
    model, tr_loader, va_loader, opt2b, sched2b,
    epochs=17, phase_name='P2b',
    save_path=MODELS_DIR / 'phase2_best.pth',
    patience=7,
)
print(f'\nPhase 2 best val macro F1 : {max(hist2["val_f1"]):.4f}')
print(f'Phase 2 best val accuracy  : {max(hist2["val_acc"]):.4f}')

In [ ]:
plot_training(hist2, 'Phase 2', PLOTS_DIR / 'phase2_training.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'phase2_training.png')); plt.axis('off'); plt.show()

## Evaluation — Load Best Phase 2 Model

In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / 'phase2_best.pth',
                                  map_location=DEVICE, weights_only=True))
model = model.to(DEVICE).eval()
print('Best Phase 2 model loaded.')

# ── Threshold comparison (Task 1) ───────────────────────────────────────────
thr_results, normal_scores, test_scores, binary_gt = threshold_comparison(
    model, val_normal_clips, test_clips, y_test, device=DEVICE)
thr_opt = _youden_threshold(test_scores, binary_gt)
print(f'ROC-optimal threshold: {thr_opt:.5f}')
_save_json(thr_results, RESULTS_DIR / 'threshold_comparison.json')

plot_roc(test_scores, binary_gt, 'ROC — Anomaly Detection (1−P(Normal))',
         PLOTS_DIR / 'roc_anomaly.png')
plot_anomaly_scores(test_scores, y_test, thr_opt, PLOTS_DIR / 'anomaly_scores.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'roc_anomaly.png')); plt.axis('off'); plt.show()
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'anomaly_scores.png')); plt.axis('off'); plt.show()

In [ ]:
# ── Classification report (Task 2) ──────────────────────────────────────────
y_pred, probs = get_predictions(model, test_clips, y_test, device=DEVICE)
report = classification_report(y_test, y_pred, target_names=CLASSES,
                               output_dict=True, zero_division=0)
print(classification_report(y_test, y_pred, target_names=CLASSES, zero_division=0))
_save_json(report, RESULTS_DIR / 'classification_report.json')

plot_confusion_matrix(y_test, y_pred, PLOTS_DIR / 'confusion_matrix.png')
plot_per_class_metrics(report, PLOTS_DIR / 'per_class_metrics.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'confusion_matrix.png')); plt.axis('off'); plt.show()
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'per_class_metrics.png')); plt.axis('off'); plt.show()

In [ ]:
# ── Binary anomaly AUC ───────────────────────────────────────────────────────
p_anom  = 1.0 - probs[:, CLASS_TO_IDX['Normal']]
auc_bin = roc_auc_score(binary_gt, p_anom)
pred_b  = (p_anom > 0.5).astype(int)
p, r, f, _ = precision_recall_fscore_support(binary_gt, pred_b, average='binary', zero_division=0)
print(f'Binary anomaly — AUC:{auc_bin:.3f}  P:{p:.3f}  R:{r:.3f}  F1:{f:.3f}')
_save_json({'auc': auc_bin, 'precision': p, 'recall': r, 'f1': f},
           RESULTS_DIR / 'binary_report.json')

plot_roc(p_anom, binary_gt, 'Binary Anomaly ROC (1−P(Normal))',
         PLOTS_DIR / 'binary_roc.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'binary_roc.png')); plt.axis('off'); plt.show()

In [ ]:
# ── Latent space (PCA + t-SNE) ───────────────────────────────────────────────
n_emb   = min(2000, len(test_clips))
embs    = get_embeddings(model, test_clips[:n_emb], device=DEVICE)
lbl_emb = y_test[:n_emb]
plot_latent_space(embs, lbl_emb, method='pca',  save_path=PLOTS_DIR / 'pca_latent.png')
plot_latent_space(embs, lbl_emb, method='tsne', save_path=PLOTS_DIR / 'tsne_latent.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'pca_latent.png'));  plt.axis('off'); plt.show()
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'tsne_latent.png')); plt.axis('off'); plt.show()

## Timeline Generation

In [ ]:
seq_clips, seq_labels, segment_info = build_test_sequence(
    test_clips, y_test, clips_per_segment=20)
print(f'Sequence: {len(seq_clips)} clips')
for cls, s, e in segment_info:
    print(f'  [{cls}] clips {s}–{e}')

pred_labels, pred_scores = sliding_window_predict(model, seq_clips, device=DEVICE)
smoothed = majority_vote_smooth(pred_labels, window=3)
segments = merge_segments(smoothed, pred_scores, threshold=thr_opt)
print_timeline(segments)

In [ ]:
plot_score_over_time(pred_scores, pred_labels, thr_opt, PLOTS_DIR / 'score_over_time.png')
plot_timeline(segments, PLOTS_DIR / 'timeline.png')
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'score_over_time.png')); plt.axis('off'); plt.show()
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'timeline.png'));         plt.axis('off'); plt.show()

## Grad-CAM Spatial Explainability

In [ ]:
# Single example: Shoplifting
fight_idx = np.where(y_test == CLASS_TO_IDX['Shoplifting'])[0]
clip_t    = ClipDataset([test_clips[fight_idx[0]]], training=False)[0]
frame_np  = _denorm_frame(clip_t[CLIP_LEN // 2])
cam, pred_cls, conf = compute_gradcam(
    model, clip_t, CLASS_TO_IDX['Shoplifting'], frame_idx=CLIP_LEN // 2, device=DEVICE)
overlay = overlay_heatmap(frame_np, cam)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(frame_np); axes[0].set_title('Input frame (Shoplifting)', fontsize=9)
axes[1].imshow(overlay)
axes[1].set_title(f'Grad-CAM  pred={IDX_TO_CLASS[pred_cls]}  conf={conf:.2f}', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'gradcam_example.png', dpi=150, bbox_inches='tight')
plt.show(); print(f"[OK] {PLOTS_DIR / 'gradcam_example.png'}")

In [ ]:
# Batch: 2 clips per anomaly class
visualize_gradcam_batch(model, test_clips, y_test, n_per_class=2,
                        save_path=PLOTS_DIR / 'gradcam_batch.png', device=DEVICE)
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR / 'gradcam_batch.png')); plt.axis('off'); plt.show()

## Ablation — TemporalPool vs LSTM

In [ ]:
print('=' * 50 + '\nABLATION — TemporalPool vs LSTM\n' + '=' * 50)

model_pool = VideoClassifier(head=TemporalPoolHead()).to(DEVICE)
model_pool.encoder.freeze_all()
opt_ab   = torch.optim.Adam(
    [p for p in model_pool.parameters() if p.requires_grad],
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
sched_ab = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ab, T_max=PHASE1_EPOCHS)

hist_ab = train_phase(
    model_pool, tr_loader, va_loader, opt_ab, sched_ab,
    epochs=PHASE1_EPOCHS, phase_name='Ablation-Pool',
    save_path=MODELS_DIR / 'ablation_pool_best.pth',
    patience=7,
)

y_pred_ab, _   = get_predictions(model_pool, test_clips, y_test, device=DEVICE)
rep_ab         = classification_report(y_test, y_pred_ab, target_names=CLASSES,
                                        output_dict=True, zero_division=0)
f1_ab          = rep_ab['macro avg']['f1-score']

y_pred_lstm, _ = get_predictions(model, test_clips, y_test, device=DEVICE)
rep_lstm       = classification_report(y_test, y_pred_lstm, target_names=CLASSES,
                                        output_dict=True, zero_division=0)
f1_lstm        = rep_lstm['macro avg']['f1-score']

print(f'\nAblation Results:')
print(f'  A: ResNet18 + TemporalPool  macro F1 = {f1_ab:.4f}')
print(f'  B: ResNet18 + LSTM (Ours)   macro F1 = {f1_lstm:.4f}')
_save_json({'temporal_pool': rep_ab, 'lstm': rep_lstm},
           RESULTS_DIR / 'ablation_report.json')

variants = ['A: ResNet18+\nTemporalPool', 'B: ResNet18+\nLSTM (Ours)']
colors   = ['#AAAAAA', '#1E88E5']
fig, ax  = plt.subplots(figsize=(6, 4))
bars = ax.bar(variants, [f1_ab, f1_lstm], color=colors, width=0.5)
for bar, val in zip(bars, [f1_ab, f1_lstm]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1.0); ax.set_ylabel('Macro F1')
ax.set_title('Ablation: Temporal Head Comparison', fontweight='bold')
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(PLOTS_DIR / 'ablation_f1.png', dpi=150, bbox_inches='tight')
plt.show(); print(f"[OK] {PLOTS_DIR / 'ablation_f1.png'}")

## Summary
All outputs saved to `outputs_resnet/`. Key files:
- `models/phase1_best.pth` — Phase 1 checkpoint
- `models/phase2_best.pth` — Phase 2 checkpoint (main model)
- `models/ablation_pool_best.pth` — TemporalPool ablation
- `plots/phase1_training.png`, `plots/phase2_training.png` — training curves
- `plots/confusion_matrix.png`, `plots/per_class_metrics.png` — classification
- `plots/roc_anomaly.png`, `plots/anomaly_scores.png` — anomaly detection
- `plots/timeline.png`, `plots/score_over_time.png` — timeline
- `plots/gradcam_example.png`, `plots/gradcam_batch.png` — explainability
- `plots/pca_latent.png`, `plots/tsne_latent.png` — latent space
- `plots/ablation_f1.png` — ablation comparison
- `results/classification_report.json`, `results/threshold_comparison.json` — metrics